In [ ]:
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import json
from utils.data_loader import load_gbs_allensbach

save_path = Path("../notebooks/diagrams/statistical_analysis")
save_path.mkdir(exist_ok=True, parents=True)

method_names = ["MRS", "Soft-MRS"]

In [ ]:
gbs_allensbach, columns, independant_variable = load_gbs_allensbach()

N = gbs_allensbach[gbs_allensbach["label"] == 1].copy()
R = gbs_allensbach[gbs_allensbach["label"] == 0].copy()

In [ ]:
mrs_sample_weights_path = "../results/statistical_analysis_mrs/gbs_allensbach/mrs-forest/sample_weights.json"
soft_mrs_sample_weights_path = "../thesis_results/statistical_analysis_mrs/gbs_allensbach/soft-mrs-linear/sample_weights.json"

with open(mrs_sample_weights_path) as file:
    mrs_sample_weights = json.load(file)

with open(soft_mrs_sample_weights_path) as file:
    soft_mrs_sample_weights = json.load(file)

In [ ]:
soft_mrs_sample_weights = np.mean(soft_mrs_sample_weights, axis=0)
mrs_sample_weights = np.mean(mrs_sample_weights, axis=0)

soft_mrs_sample_weights_std = np.std(soft_mrs_sample_weights, axis=0)
mrs_sample_weights_std = np.std(mrs_sample_weights, axis=0)

In [ ]:
min_val = N.Resilienz.min()
max_val = N.Resilienz.max()
val_width = max_val - min_val
bins = 10
bin_width = val_width / bins

fig, axes = plt.subplots(1, 4, sharey=True, sharex=True, figsize=(10, 5))
sns.histplot(
    N, x="Resilienz", bins=bins, stat="probability", kde=True, ax=axes[0],
).set(title="GBS", xlabel="", ylabel="Density")
sns.histplot(
    R, x="Resilienz", bins=bins, stat="probability", kde=True, ax=axes[1],
).set(title="Allensbach", xlabel="", ylabel="Density")
sns.histplot(
    N,
    x="Resilienz",
    weights=mrs_sample_weights,
    bins=bins,
    stat="probability",
    kde=True,
    ax=axes[2],
).set(title="MRS", xlabel="", ylabel="Density")
sns.histplot(
    N,
    x="Resilienz",
    weights=soft_mrs_sample_weights,
    bins=bins,
    stat="probability",
    kde=True,
    ax=axes[3],
).set(title="Soft-MRS", xlabel="", ylabel="Density")
fig.supxlabel("Brief Resilience Scale")
plt.xticks(np.arange(1, 5 + bin_width, 2 *  bin_width))


fig.savefig(save_path / "resilience_histograms.pdf")

In [ ]:
plt.figure(figsize=(10, 5))
sns.ecdfplot(N, x="Resilienz", label="GBS")
sns.ecdfplot(R, x="Resilienz", label="Allensbach", linestyle="dashed")
sns.ecdfplot(
    N,
    x="Resilienz",
    weights=mrs_sample_weights,
    label="MRS",
    linestyle="dotted",
)
sns.ecdfplot(
    N,
    x="Resilienz",
    weights=soft_mrs_sample_weights,
    label="Soft-MRS",
    linestyle="dashdot",
).set(xlabel="Brief Resilience Scale")
plt.legend()
plt.savefig(save_path / "empirical_cumulative_distribution_functions.pdf")